In [ ]:
# Extract highres site predictions from 4_downscale/results_highres_uk/ NetCDF files.
#
# For each (variable, scenario, site):
#   - reads both modhighice and modlowice predictions for all 90 members
#   - selects ice state per timestep using GSL (ice < 0 → highice, else → lowice)
#   - subtracts PI-control value to produce anomaly
#   - for pr: also converts units from kg m⁻² s⁻¹ → mm month⁻¹
#   - saves flat to site_data/highres_results/ with standard header
#     output shape: (90 members × 1001 timesteps)
#
# Output filename: {var}_{scenario}_UK_site{N}.txt
# Run from: 4_downscale/get_site_txt/


In [ ]:
import os
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path

# ── Server paths (BluePebble) ─────────────────────────────────────────────
INPUT_DIR    = Path("/user/work/bo20541/downscaling_NWS/results_highres_uk")
TRAINING_DIR = Path("/user/work/bo20541/downscaling_NWS/training_data_highres")
GSL_DIR      = Path("/user/home/bo20541/NWS/downscaling_NWS/emul_inputs_updatedCO2")
OUTPUT_DIR   = Path("/user/work/bo20541/downscaling_NWS/site_data_highres_UK")
SITES_FILE   = Path("UK_sites_index.res")  # same dir as this notebook

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNIT_FACTOR = 60.0 * 60.0 * 24.0 * 1000.0 * 30.0  # kg m⁻² s⁻¹ → mm month⁻¹

META = {
    "tas": {"long_name": "2m Air Temperature",   "units": "degC"},
    "pr":  {"long_name": "Surface Precipitation", "units": "mm month^-1"},
}

HEADER_TMPL = (
    "# Variable: {var}\n"
    "# Long name: {long_name}\n"
    "# Units: {units}\n"
    "# Scenario: {scenario}\n"
    "# Site: {site}\n"
    "# Shape: 90 ensemble members x 1001 time steps\n"
    "# Rows: ensemble members (1-90); Columns: time steps (0 to 1000 kyr AP)\n"
    "# Ice-state selection: GSL<0 -> modhighice, GSL>=0 -> modlowice\n"
    "# Resolution: high (CHELSA downscaled, 360x720)\n"
)

# ── Config ────────────────────────────────────────────────────────────────
vars_list = ["tas", "pr"]
scenarios = ["SSP126", "SSP245", "SSP370", "SSP585", "10000PGC"]
# Note: highres downscaling does not include the natural scenario
members   = range(1, 91)
sites     = pd.read_csv(SITES_FILE)

# ── Main loop ─────────────────────────────────────────────────────────────
for var in vars_list:
    # Load PI-control value (timestep index 60 = PI year) from training data
    pi_ds    = xr.open_dataset(TRAINING_DIR / f"emul_in_Y_modhighice_{var}.nc")
    pi_field = pi_ds["var"][60, :, :].values   # (lat, lon)
    pi_lat   = pi_ds["lat"].values
    pi_lon   = pi_ds["lon"].values
    pi_ds.close()
    print(f"[{var}] PI field: shape={pi_field.shape}  "
          f"min={pi_field.min():.4f}  max={pi_field.max():.4f}")

    for scenario in scenarios:
        for _, site_row in sites.iterrows():
            lati     = int(site_row["lat_index"])
            loni     = int(site_row["lon_index"])
            lat_val  = float(site_row["lat"])
            lon_val  = float(site_row["lon"])
            site_num = int(site_row["site_num"])
            out_file = OUTPUT_DIR / f"{var}_{scenario}_UK_site{site_num}.txt"

            if out_file.exists():
                with open(out_file) as fh:
                    if fh.readline().startswith("#"):
                        print(f"  SKIP {out_file.name} (already done)")
                        continue

            # PI value at this site
            ilat    = int(np.argmin(np.abs(pi_lat - lat_val)))
            ilon    = int(np.argmin(np.abs(pi_lon - lon_val)))
            pi_site = float(pi_field[ilat, ilon])

            member_series = []
            for member in members:
                # Load GSL (ice column) for this member
                gsl_file = GSL_DIR / f"emul_inputs_{scenario}.{member}.updated.res"
                emul_df  = pd.read_csv(gsl_file, sep=r"\s+", engine="python")
                ice_col  = next(c for c in emul_df.columns if str(c).lower() == "ice")
                ice      = emul_df[ice_col].to_numpy()  # (1001,)

                # Load highice and lowice predictions
                def load_site(state):
                    nc = INPUT_DIR / f"{var}_{state}_highres_{scenario}_{member}_UK.nc"
                    with xr.open_dataset(nc) as ds:
                        return ds["var"][:, lati, loni].values  # (1001,)

                high_vals = load_site("modhighice")
                low_vals  = load_site("modlowice")

                if len(ice) != len(high_vals):
                    raise ValueError(
                        f"Length mismatch member {member}: ice={len(ice)}, data={len(high_vals)}"
                    )

                # Select ice state: GSL < 0 → highice, else → lowice
                vals = np.where(ice < 0, high_vals, low_vals)
                member_series.append(vals)

            # Stack → (90 members, 1001 timesteps)
            data = np.stack(member_series)

            # Compute anomaly relative to PI
            if var == "tas":
                data = data - pi_site
            elif var == "pr":
                data = (data - pi_site) * UNIT_FACTOR  # → mm month⁻¹

            # Write with standard header (90 rows × 1001 cols)
            header = HEADER_TMPL.format(
                var=var, long_name=META[var]["long_name"],
                units=META[var]["units"], scenario=scenario, site=site_num,
            )
            with open(out_file, "w") as fh:
                fh.write(header)
                np.savetxt(fh, data, fmt="%.6f")

            print(f"  Saved {out_file.name}  shape={data.shape}")

print("\nAll done.")
